In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [ ]:
cropdata = pd.read_csv("/content/Agri_model_cleaned.csv")

In [ ]:
cropdata.describe()

In [ ]:
cropdata.info()

In [ ]:

print(cropdata.dtypes.value_counts())

#Check Missing Values

In [ ]:
missing = (cropdata.isna().mean() * 100).sort_values(ascending=False)
print("\nMissing Values (%):")
print(missing[missing > 0])

In [ ]:
numeric_cols = cropdata.select_dtypes(include=[np.number]).columns
print("\nValue Counts for Suspicious Columns:")
for col in ['pH', 'EC', 'N', 'P', 'K']:
    if col in cropdata.columns:
        print(f"\n{col}: {cropdata[col].value_counts().head()}")

#Feature Extraction

In [ ]:

# Strip whitespace from column names
cropdata.columns = cropdata.columns.str.strip()

# 1) Soil Nutrient Index
cropdata["SNI"] = cropdata["N"] + cropdata["P"] + cropdata["K"]

# 2) Micronutrient Index
cropdata["MNI"] = cropdata["Fe"] + cropdata["Cu"] + cropdata["B"] + cropdata["Mo"]

# 3) Vegetation Health Index
cropdata["VHI"] = (cropdata["NDVI"] + cropdata["EVI"]) / 2

# 4) Stress Index
cropdata["Stress_Index"] = abs(cropdata["NDVI"] - cropdata["EVI"])

# 5) Temperature-Humidity Index (THI)
cropdata["THI"] = cropdata["Temperature"] - (
    0.55 * (1 - cropdata["Humidity"] / 100) * (cropdata["Temperature"] - 14.5)
)

# 6) Rain-adjusted irrigation
cropdata["Irrigation_Adjusted"] = cropdata["Irrigation_Frequency"] / (cropdata["Rainfall"] + 1)

# 7) NPK Ratios
cropdata["NP_ratio"] = cropdata["N"] / (cropdata["P"] + 1)
cropdata["NK_ratio"] = cropdata["N"] / (cropdata["K"] + 1)
cropdata["PK_ratio"] = cropdata["P"] / (cropdata["K"] + 1)

# 8) Year normalized
cropdata["Years_passed"] = cropdata["Year"] - cropdata["Year"].min()

# 9) Interaction feature
cropdata["Water_Nitrogen"] = cropdata["Rainfall"] * cropdata["N"]

# 10) One-Hot Encoding for categorical features
cropdata = pd.get_dummies(
    cropdata,
    columns=["Soil_Type", "Crop_Type", "Fertilizer_Type", "Pesticide_Usage", "Region", "Season"],
    drop_first=True
)

# Save processed dataset
cropdata.to_csv("processed_cropdata.csv", index=False)

cropdata.head()


In [ ]:

# Select Features & Target

X = cropdata.drop("Yield", axis=1)
y = cropdata["Yield"]
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:


for col in ['NDVI','EVI','Temperature','Rainfall','pH','EC','N','P','K']:
    Q1 = cropdata[col].quantile(0.25)
    Q3 = cropdata[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    cropdata = cropdata[(cropdata[col] >= lower) & (cropdata[col] <= upper)]



In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
cropdata = pd.get_dummies(cropdata, drop_first=True)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared: {r2:.4f}")

#Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import ExtraTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB # Import GaussianNB
from sklearn.neighbors import KNeighborsClassifier # Import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier # Import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier # Import GradientBoostingClassifier


results = {}

# Discretize the continuous 'Yield' into categories for classification
# Define bins for Yield (e.g., 3 categories: Low, Medium, High)
num_bins = 3
y_train_cat = pd.cut(y_train, bins=num_bins, labels=False, include_lowest=True)
y_test_cat = pd.cut(y_test, bins=num_bins, labels=False, include_lowest=True)

models = {
    'LogisticRegression': LogisticRegression(max_iter=2000),
    'GaussianNB': GaussianNB(),
    'SVC': SVC(),
    'KNeighborsClassifier': KNeighborsClassifier(),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'ExtraTreeClassifier': ExtraTreeClassifier(),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42),
    'BaggingClassifier': BaggingClassifier(),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'AdaBoostClassifier': AdaBoostClassifier()
}

results = []
for name, model in models.items():
    print(f"\n============================ {name} ============================\n")

    # Train using discretized y
    model.fit(X_train, y_train_cat)

    # Predict
    preds = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test_cat, preds)
    prec = precision_score(y_test_cat, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test_cat, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test_cat, preds, average='weighted', zero_division=0)

    # Save results
    results.append([name, acc, prec, rec, f1])

    # Print classification report
    print("Classification Report:\n")
    print(classification_report(y_test_cat, preds, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(y_test_cat, preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()
results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])
print("\n\n================== MODEL PERFORMANCE COMPARISON ==================\n")
print(results_df)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------- Load Data -------------------
cropdata = pd.read_csv("/content/processed_cropdata.csv")

# ------------------- Clean Column Names -------------------
cropdata.columns = cropdata.columns.str.strip()
cropdata.columns = cropdata.columns.str.replace(' ', '_')

# ------------------- Remove Outliers -------------------
num_cols = ['NDVI','EVI','Temperature','Humidity','Rainfall','pH','EC','N','P','K','Ca','Mg','Fe','Cu','B','Mo','OC']
for col in num_cols:
    if col in cropdata.columns:
        Q1 = cropdata[col].quantile(0.25)
        Q3 = cropdata[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5*IQR
        upper = Q3 + 1.5*IQR
        cropdata = cropdata[(cropdata[col]>=lower) & (cropdata[col]<=upper)]

# ------------------- Feature Engineering -------------------
fe_cols = ['N','P','K','NDVI','EVI','Temperature','Humidity','Rainfall']
if all(x in cropdata.columns for x in fe_cols):
    cropdata['SNI'] = cropdata['N'] + cropdata['P'] + cropdata['K']
    cropdata['NP_ratio'] = cropdata['N'] / (cropdata['P']+1)
    cropdata['NK_ratio'] = cropdata['N'] / (cropdata['K']+1)
    cropdata['PK_ratio'] = cropdata['P'] / (cropdata['K']+1)
    cropdata['VHI'] = (cropdata['NDVI'] + cropdata['EVI']) / 2
    cropdata['Stress_Index'] = abs(cropdata['NDVI'] - cropdata['EVI'])
    cropdata['THI'] = cropdata['Temperature'] - (0.55*(1 - cropdata['Humidity']/100)*(cropdata['Temperature']-14.5))
    cropdata['Water_Nitrogen'] = cropdata['Rainfall'] * cropdata['N']

# ------------------- One-Hot Encoding -------------------
cat_cols = ['Soil_Type', 'Crop_Type', 'Fertilizer_Type','Pesticide_Usage','Region','Season']
existing_cat_cols = [col for col in cat_cols if col in cropdata.columns]
cropdata = pd.get_dummies(cropdata, columns=existing_cat_cols, drop_first=True)

# ------------------- Features & Target -------------------
X = cropdata.drop('Yield', axis=1)
y = cropdata['Yield']

# ------------------- Scaling -------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ------------------- Train-Test Split -------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ------------------- Define Models -------------------
models = {
    'RandomForest': RandomForestRegressor(n_estimators=500, max_depth=25, random_state=42),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=500, max_depth=25, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=400, max_depth=10, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42),
    'NeuralNetwork': MLPRegressor(hidden_layer_sizes=(128,64,32), max_iter=500, activation='relu', random_state=42)
}

# ------------------- Train & Evaluate -------------------
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    results[name] = r2
    print(f"{name} R2 Score: {r2:.4f}, MSE: {mse:.4f}")

    # Plot Predicted vs True
    plt.figure(figsize=(6,5))
    plt.scatter(y_test, preds, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel("True Yield")
    plt.ylabel("Predicted Yield")
    plt.title(f"{name} Predicted vs True Yield")
    plt.show()

# ------------------- Summary Table -------------------
results_df = pd.DataFrame(list(results.items()), columns=['Model','R2_Score']).sort_values(by='R2_Score', ascending=False)
print("\n=== Model R2 Score Comparison ===")
print(results_df)


In [ ]:
from sklearn.preprocessing import LabelEncoder

# ------------------- Convert Yield Classes to Numeric -------------------
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # Low -> 0, Medium -> 1, High -> 2

# ------------------- Train-Test Split -------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# ------------------- Scaling -------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------- Models -------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

models = {
    'RandomForest': RandomForestClassifier(n_estimators=500, max_depth=25, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=400, max_depth=10, learning_rate=0.05, random_state=42),
    'NeuralNetwork': MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=500, random_state=42)
}

# ------------------- Train & Evaluate -------------------
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    print(f"{name} Accuracy: {acc:.4f}")

    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()


In [ ]:
# ============================================================
# CELL 1: Import
# ============================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              BaggingClassifier, ExtraTreesClassifier, VotingClassifier,
                              StackingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# CELL 2: Load & Clean Data
# ============================================================
crop_fulldata = pd.read_csv("/content/processed_cropdata.csv")

# Remove outliers using IQR
def remove_outliers(df, columns):
    df_clean = df.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
    return df_clean

numeric_cols = crop_fulldata.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('Yield') if 'Yield' in numeric_cols else None

print(f"Before outlier removal: {crop_fulldata.shape}")
# Apply outlier removal to all relevant numeric columns, not just top 10
crop_fulldata = remove_outliers(crop_fulldata, [col for col in numeric_cols if col not in ['Year']])
print(f"After outlier removal: {crop_fulldata.shape}")


# ============================================================
# CELL 3: Create Balanced Yield Classes
# ============================================================
crop_fulldata['Yield_Class'] = pd.qcut(crop_fulldata['Yield'],
                                       q=3,
                                       labels=['Low', 'Medium', 'High'],
                                       duplicates='drop')

print("\nYield Class Distribution:")
print(crop_fulldata['Yield_Class'].value_counts())


# ============================================================
# CELL 4: Define Features and Target
# ============================================================
X = crop_fulldata.drop(['Yield', 'Yield_Class'], axis=1, errors='ignore')
y = crop_fulldata['Yield_Class']

# NPK features - assuming these are already in X from previous feature engineering
# If not, they would need to be re-created here or ensured to be present.
# The previous cell XAPneZl2hPtg created these features:
# SNI, MNI, VHI, Stress_Index, THI, Irrigation_Adjusted, NP_ratio, NK_ratio, PK_ratio, Years_passed, Water_Nitrogen
# We will only keep feature engineering that relies on existing columns.
X['NPK_Total'] = X['N'] + X['P'] + X['K']
X['NPK_Balance'] = X['N'] / (X['P'] + X['K'] + 1e-10)
X['NK_Ratio'] = X['N'] / (X['K'] + 1e-10)
X['PK_Ratio'] = X['P'] / (X['K'] + 1e-10)

# Climate features (assuming Temperature, Humidity, Rainfall exist)
X['Climate_Score'] = (X['Temperature'] * 0.4 + X['Humidity'] * 0.3 + X['Rainfall'] * 0.3)
X['Temp_Humidity'] = X['Temperature'] * X['Humidity']
X['Temp_Rainfall'] = X['Temperature'] * X['Rainfall']

# Vegetation features (assuming NDVI, EVI exist)
X['Veg_Health'] = (X['NDVI'] + X['EVI']) / 2
X['NDVI_EVI'] = X['NDVI'] * X['EVI']

# Micronutrients (Fe, Cu, B, Mo assumed to exist)
X['Micro_Total'] = X['Fe'] + X['Cu'] + X['B'] + X['Mo']


print(f"Total features: {X.shape[1]}")

# Encode y
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)


# ============================================================
# CELL 5: Split with Stratification
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.15,  # Smaller test set = more training data
    random_state=42,
    stratify=y_encoded
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


# ============================================================
# CELL 6: Robust Scaling
# ============================================================
scaler = RobustScaler()  # Better for outliers than StandardScaler
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# CELL 7: Optimized Models
# ============================================================
models = {
    'GaussianNB': GaussianNB(var_smoothing=1e-9),
    'RandomForest': RandomForestClassifier(
        n_estimators=1000, max_depth=30, min_samples_split=2,
        min_samples_leaf=1, max_features='sqrt', random_state=42, n_jobs=-1
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=1000, max_depth=30, min_samples_split=2,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=1000, max_depth=12, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        reg_lambda=0.1, random_state=42, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=1000, max_depth=12, learning_rate=0.05,
        num_leaves=31, random_state=42, n_jobs=-1, verbose=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=10,
        subsample=0.8, random_state=42
    ),
    'SVC': SVC(
        kernel='rbf', C=100, gamma='scale', probability=True, random_state=42
    ),
    'Bagging': BaggingClassifier(
        n_estimators=500, max_samples=0.8, max_features=0.8, random_state=42, n_jobs=-1
    )
}


# ============================================================
# CELL 8: Train & Evaluate
# ============================================================
results = {}
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:20s} Accuracy: {acc:.6f}")

results_sorted = dict(sorted(results.items(), key=lambda x: x[1], reverse=True))


# ============================================================
# CELL 9: Stacking Ensemble (Best Method!)
# ============================================================
print("\n" + "="*60)
print("STACKING ENSEMBLE")
print("="*60)

# Base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=1000, max_depth=30, random_state=42, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=1000, max_depth=12, learning_rate=0.05, random_state=42, n_jobs=-1)),
    ('lgb', LGBMClassifier(n_estimators=1000, max_depth=12, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1)),
    ('et', ExtraTreesClassifier(n_estimators=1000, max_depth=30, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(n_estimators=500, learning_rate=0.05, max_depth=10, random_state=42))
]

# Meta model
meta_model = LogisticRegression(C=10, max_iter=1000, random_state=42)

# Stacking
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

stacking_clf.fit(X_train_scaled, y_train)
y_pred_stack = stacking_clf.predict(X_test_scaled)
stack_acc = accuracy_score(y_test, y_pred_stack)

print(f"\nStacking Classifier Accuracy: {stack_acc:.6f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_stack)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])
plt.title(f'Stacking - Confusion Matrix (Acc: {stack_acc:.4f})',
         fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()


# ============================================================
# CELL 10: Voting Ensemble (Soft)
# ============================================================
print("\n" + "="*60)
print("VOTING ENSEMBLE")
print("="*60)

voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=1000, max_depth=30, random_state=42, n_jobs=-1)),
        ('xgb', XGBClassifier(n_estimators=1000, max_depth=12, learning_rate=0.05, random_state=42, n_jobs=-1)),
        ('lgb', LGBMClassifier(n_estimators=1000, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1)),
        ('nb', GaussianNB()),
        ('svc', SVC(kernel='rbf', C=100, probability=True, random_state=42)),
        ('et', ExtraTreesClassifier(n_estimators=1000, random_state=42, n_jobs=-1))
    ],
    voting='soft',
    n_jobs=-1
)

voting_clf.fit(X_train_scaled, y_train)
y_pred_vote = voting_clf.predict(X_test_scaled)
vote_acc = accuracy_score(y_test, y_pred_vote)

print(f"\nVoting Classifier Accuracy: {vote_acc:.6f}")


# ============================================================
# CELL 11: Cross-Validation
# ============================================================
print("\n" + "="*60)
print("CROSS-VALIDATION (10-Fold)")
print("="*60)

best_models = {
    'RandomForest': RandomForestClassifier(n_estimators=1000, max_depth=30, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=1000, max_depth=12, learning_rate=0.05, random_state=42, n_jobs=-1),
    'GaussianNB': GaussianNB(),
    'Stacking': stacking_clf
}

for name, model in best_models.items():
    if name != 'Stacking':  # Stacking already trained
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring='accuracy', n_jobs=-1)
        print(f"{name:15s} CV: {cv_scores.mean():.6f} (+/- {cv_scores.std():.6f})")


# ============================================================
# CELL 12: Final Comparison
# ============================================================
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

all_results = {
    **results,
    'Stacking_Ensemble': stack_acc,
    'Voting_Ensemble': vote_acc
}

all_sorted = dict(sorted(all_results.items(), key=lambda x: x[1], reverse=True))

print("\nAll Models Ranked:")
for i, (name, acc) in enumerate(all_sorted.items(), 1):
    emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
    print(f"{emoji} {i:2d}. {name:25s} {acc:.6f}")

print(f"\n🏆 BEST: {list(all_sorted.keys())[0]} with {list(all_sorted.values())[0]:.6f} accuracy")


# ============================================================
# CELL 13: Classification Report
# ============================================================
best_pred = y_pred_stack if stack_acc >= vote_acc else y_pred_vote
best_name = "Stacking" if stack_acc >= vote_acc else "Voting"

print(f"\n{best_name} Ensemble - Classification Report:")
print(classification_report(y_test, best_pred,
                          target_names=['Low', 'Medium', 'High']))


# ============================================================
# CELL 14: Visualization
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
names = list(all_sorted.keys())[:10]
scores = list(all_sorted.values())[:10]
colors = plt.cm.RdYlGn(np.linspace(0.3, 1, len(names)))

bars = ax1.barh(names, scores, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Top 10 Models Comparison', fontsize=14, fontweight='bold')
ax1.set_xlim([0.9, 1.0])
ax1.grid(axis='x', alpha=0.3)

for i, (bar, score) in enumerate(zip(bars, scores)):
    ax1.text(score - 0.005, i, f'{score:.4f}', va='center', ha='right',
            fontsize=10, fontweight='bold', color='white')

# Confusion matrix
cm_best = confusion_matrix(y_test, best_pred)
sns.heatmap(cm_best, annot=True, fmt='d', cmap='YlGnBu', ax=ax2,
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'],
            cbar_kws={'label': 'Count'})
ax2.set_title(f'{best_name} - Confusion Matrix', fontsize=14, fontweight='bold')
ax2.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax2.set_ylabel('True', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Pipeline Complete! Check if accuracy >= 99%")